# Traffic Feature Engineering
This notebook calculates weighted density scores (PCUs) and extracts temporal features from timestamps to feed models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../datasets/raw/TrafficTwoMonth.csv')
df.head()

## Extract Datetime Elements
We extract Hour, Minute, Day of Week, Weekend indicator, and Peak Hour flags.

In [ ]:
# Hour and Minute extraction
df['Hour'] = pd.to_datetime(df['Time'], format='%I:%M:%S %p').dt.hour
df['Minute'] = pd.to_datetime(df['Time'], format='%I:%M:%S %p').dt.minute

# Peak Hour (7:00-9:30 AM, 4:30-7:30 PM)
df['TimeVal'] = df['Hour'] + df['Minute'] / 60.0
df['IsPeakHour'] = (((df['TimeVal'] >= 7.0) & (df['TimeVal'] <= 9.5)) | 
                    ((df['TimeVal'] >= 16.5) & (df['TimeVal'] <= 19.5))).astype(int)

# Day of week mapping
day_map = {'Monday':0, 'Tuesday':1, 'Wednesday':2, 'Thursday':3, 'Friday':4, 'Saturday':5, 'Sunday':6}
df['DayOfWeek'] = df['Day of the week'].map(day_map)
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

df[['Time', 'Hour', 'IsPeakHour', 'DayOfWeek', 'IsWeekend']].head(10)

## Passenger Car Unit (PCU) Weighting
We assign PCU weights representing road occupancy: Cars=1, Bikes=0.2, Buses=2.5, Trucks=3.0.

In [ ]:
df['DensityScore'] = (
    df['CarCount'] * 1.0 +
    df['BikeCount'] * 0.2 +
    df['BusCount'] * 2.5 +
    df['TruckCount'] * 3.0
)

plt.figure(figsize=(10, 6))
sns.boxplot(x='Traffic Situation', y='DensityScore', data=df, order=['low', 'normal', 'high', 'heavy'])
plt.title('Passenger Car Unit (PCU) Density Score vs Traffic Situation')
plt.show()